In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_cohere import ChatCohere
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages

In [2]:
load_dotenv()

True

In [3]:
# create the model object.
model = ChatCohere(
    model = 'command-a-03-2025', 
    temperature = 0.7
)

In [4]:
# define the state.
class ChatState(TypedDict):

    messages : Annotated[list[BaseMessage], add_messages]

In [5]:
# define the node.
def chat_node(state : ChatState):

    # fetch the message for query.
    messages = state['messages']

    # invoke the function.
    response = model.invoke(messages)

    # update the state.
    return {
        'messages' : [response]
    }

In [14]:
# define the graph.
graph = StateGraph(ChatState)

# add the nodes in the graph.
graph.add_node('chat_node', chat_node)

# add the edges in the graph.
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

# compile the graph.
chatbot = graph.compile(checkpointer = checkPointer)

In [7]:
# define the initial state.
initial_state = {
    'messages' : [HumanMessage(content = 'What is the capital of India?')]
}

# invoke the entire workflow.
response = chatbot.invoke(initial_state)

# print the response.
print(response)

{'messages': [HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='58b08e76-4cff-480c-ba00-2a62140355bc'), AIMessage(content='The capital of India is **New Delhi**. It is the seat of the Government of India and is located within the National Capital Territory of Delhi. New Delhi is a major cultural, political, and commercial center in India.', additional_kwargs={'id': '9e9bfd42-4da4-4178-9a8f-263f3348e1ac', 'finish_reason': 'COMPLETE', 'token_count': {'input_tokens': 502.0, 'output_tokens': 47.0}}, response_metadata={'id': '9e9bfd42-4da4-4178-9a8f-263f3348e1ac', 'finish_reason': 'COMPLETE', 'token_count': {'input_tokens': 502.0, 'output_tokens': 47.0}}, id='lc_run--01a0c766-28e9-70c1-a0ae-98ceb66a24d0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 502, 'output_tokens': 47, 'total_tokens': 549})]}


In [8]:
response['messages'][-1].content

'The capital of India is **New Delhi**. It is the seat of the Government of India and is located within the National Capital Territory of Delhi. New Delhi is a major cultural, political, and commercial center in India.'

## We are Writing the ChatBot messages workflow Code.

In [9]:
while True:

    user_message = input('Type your query : ')

    print('User : ', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    response = chatbot.invoke({
        'messages' : HumanMessage(content = user_message)
    })

    print('AI : ', response['messages'][-1].content)

User :  Hi
AI :  Hello! How can I assist you today? I'm here to help with any questions or tasks you might have. Whether it's information, creative writing, or problem-solving, feel free to let me know what you need.
User :  My name is Uzaif
AI :  Hello, Uzaif! It's nice to meet you. How can I assist you today? Whether you have questions, need information, or just want to chat, feel free to let me know!
User :  What is my name?
AI :  I don't have access to personal information, so I don't know your name. If you'd like to tell me, I'd be happy to address you by it!
User :  add 20 to 40
AI :  Sure! 

40 + 20 = 60
User :  now add 20 to the result
AI :  I'm unable to add 20 to a result since there is no previous result provided. Please provide the initial value or result, and I'll be happy to add 20 to it.
User :  exit


### Adding Persistance concept in the chatbot, because it will not get the memory of the previous chat.

In [10]:
# As we know that the above chatbot does not have a capability to restore the previous chats, so used persistance concept here.

In [11]:
from langgraph.checkpoint.memory import MemorySaver

In [13]:
thread_id = 1
checkPointer = MemorySaver()

In [ ]:
while True:

    user_message = input('Type your query : ')

    print('User : ', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {
        'configurable' : {'thread_id' : thread_id}
    }    

    response = chatbot.invoke({
        'messages' : HumanMessage(content = user_message)
    }, config = config)

    print('AI : ', response['messages'][-1].content)

User :  HI, my name is Mohd Uzaif
AI :  Hello Mohd Uzaif! It's nice to meet you. How can I assist you today? Whether you have questions, need information, or just want to chat, I'm here to help.
User :  Now tell me my name
AI :  Your name is **Mohd Uzaif**. How can I assist you further?
User :  add 20 with 30
AI :  Sure! Here's the calculation:

20 + 30 = 50

So, the result is **50**.
User :  now add 40 with the result
AI :  Sure! Let's add 40 to the previous result of 50:

50 + 40 = 90

So, the new result is **90**.
User :  exit
